# Лабораторная работа 1, Самсонов Савелий Артёмович М8О-406Б-21

### Выбор задач и датасетов

Киноиндустрия сталкивается с серьезной проблемой при прогнозировании успеха фильмов на конкурентном рынке. Понимание ключевых факторов, влияющих на это и на получение дохода от фильма, имеет решающее значение для продюсеров, студий и инвесторов при принятии стратегических решений. Поэтому для определения переменных, влияющих на успех фильма, таких как бюджет, маркетинговые расходы, продолжительность фильма и рейтинги главных актеров, режиссеров и критиков, необходимо прогностическое моделирование. Для обучения моделей, которые могут предсказать успешность фильмов в прокате, взяты датасеты Movie_classification.csv и Movie_regression.xls, опубликованные на kaggle.

### Выбор метрик для классификации

1. Accuracy, измеряет долю правильно классифицированных экземпляров от общего числа примеров. Простая и понятная метрика, но может давать ошибочное представление для несбалансированных классов (когда классы имеют существенно отличающееся количество экземпляров).
2. Precision, измеряет долю правильных положительных предсказаний среди всех предсказанных положительных примеров.
3. Recall, измеряет долю правильно предсказанных положительных примеров среди всех реальных положительных примеров.
Precision, Recall важны в случае несбалансированных классов и при необходимости минимизировать ложные срабатывания.
4. F1-мера, является гармоническим средним между точностью и полнотой и используется для сбалансирования этих двух метрик. Комбинирует точность и полноту, идеально подходит для несбалансированных задач.

### Выбор метрик для регрессии

1. R², отношение между суммой квадратов отклонений предсказанных значений от среднего значения и суммой квадратов отклонений истинных значений от среднего. Показывает, какая доля вариации в целевой переменной объясняется моделью. Хороший показатель R² близкий к 1 означает, что модель хорошо объясняет данные, однако для некоторых типов задач (например, с незначительными отклонениями) значение R² может быть не таким информативным.
2. MAE, измеряет среднее абсолютное отклонение между предсказанными и истинными значениями. Полезна, когда важно понять, насколько в среднем модель ошибается по величине предсказанных значений. MAE не так чувствительна к выбросам, как другие метрики.
3. MSE, измеряет средний квадрат разницы между предсказанными и истинными значениями. MSE часто используется, когда важно акцентировать внимание на больших ошибках. Более чувствительна к выбросам, чем MAE, и может быть полезна, когда крупные ошибки особенно нежелательны.

In [1]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score, r2_score, mean_absolute_error, mean_squared_error
from sklearn.preprocessing import StandardScaler

In [2]:
import os
dataset_path = '.\\input'

for dirname, _, filenames in os.walk(dataset_path):
    for filename in filenames:
        print(os.path.join(dirname, filename))

.\input\Movie_classification.csv
.\input\Movie_regression.xls


## 2. Создание бейзлайна и оценка качества

### Обучение модели из sklearn (для классификации) и оценка качества по выбранным метрикам

Загрузим датасет

In [3]:
df = pd.read_csv(dataset_path + "\\Movie_classification.csv")
df

,Marketing expense,Production expense,Multiplex coverage,Budget,Movie_length,Lead_ Actor_Rating,Lead_Actress_rating,Director_rating,Producer_rating,Critic_rating,Trailer_views,3D_available,Time_taken,Twitter_hastags,Genre,Avg_age_actors,Num_multiplex,Collection,Start_Tech_Oscar
0,20.1264,59.62,0.462,36524.125,138.7,7.825,8.095,7.910,7.995,7.94,527367,YES,109.60,223.840,Thriller,23,494,48000,1
1,20.5462,69.14,0.531,35668.655,152.4,7.505,7.650,7.440,7.470,7.44,494055,NO,146.64,243.456,Drama,42,462,43200,0
2,20.5458,69.14,0.531,39912.675,134.6,7.485,7.570,7.495,7.515,7.44,547051,NO,147.88,2022.400,Comedy,38,458,69400,1
3,20.6474,59.36,0.542,38873.890,119.3,6.895,7.035,6.920,7.020,8.26,516279,YES,185.36,225.344,Drama,45,472,66800,1
4,21.3810,59.36,0.542,39701.585,127.7,6.920,7.070,6.815,7.070,8.26,531448,NO,176.48,225.792,Drama,55,395,72400,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
501,21.2526,78.86,0.427,36624.115,142.6,8.680,8.775,8.620,8.970,6.80,492480,NO,186.96,243.584,Action,27,561,44800,0
502,20.9054,78.86,0.427,33996.600,150.2,8.780,8.945,8.770,8.930,7.80,482875,YES,132.24,263.296,Action,20,600,41200,0
503,21.2152,78.86,0.427,38751.680,164.5,8.830,8.970,8.855,9.010,7.80,532239,NO,109.56,243.824,Comedy,31,576,47800,0
504,22.1918,78.86,0.427,37740.670,162.8,8.730,8.845,8.800,8.845,6.80,496077,YES,158.80,303.520,Comedy,47,607,44000,0


Удалим некоторые параметры

In [4]:
if "Genre" in df:
  del df["Genre"]
if "3D_available" in df:
  del df["3D_available"]
if "Time_taken" in df:
  del df["Time_taken"]

Просмотрим информацию о значениях полей и убедимся, что все они допустимы

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 506 entries, 0 to 505
Data columns (total 16 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Marketing expense    506 non-null    float64
 1   Production expense   506 non-null    float64
 2   Multiplex coverage   506 non-null    float64
 3   Budget               506 non-null    float64
 4   Movie_length         506 non-null    float64
 5   Lead_ Actor_Rating   506 non-null    float64
 6   Lead_Actress_rating  506 non-null    float64
 7   Director_rating      506 non-null    float64
 8   Producer_rating      506 non-null    float64
 9   Critic_rating        506 non-null    float64
 10  Trailer_views        506 non-null    int64  
 11  Twitter_hastags      506 non-null    float64
 12  Avg_age_actors       506 non-null    int64  
 13  Num_multiplex        506 non-null    int64  
 14  Collection           506 non-null    int64  
 15  Start_Tech_Oscar     506 non-null    int

Создадим выборки для обучения и тестирования

In [7]:
X1 = df.drop('Start_Tech_Oscar', axis=1)
y1 = df['Start_Tech_Oscar']

X1_train,X1_test,y1_train,y1_test = train_test_split(X1.values, y1.values, random_state = 0)

Обучение модели KNN для классификации

In [8]:
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor

sk_knn_clf = KNeighborsClassifier()
sk_knn_clf.fit(X1_train, y1_train)
sk_knn_clf_pred_res = sk_knn_clf.predict(X1_test)
sk_knn_clf_accuracy = accuracy_score(y1_test, sk_knn_clf_pred_res)
sk_knn_clf_precision = precision_score(y1_test, sk_knn_clf_pred_res)
sk_knn_clf_recall = recall_score(y1_test, sk_knn_clf_pred_res)
sk_knn_clf_f1 = f1_score(y1_test, sk_knn_clf_pred_res)

print(f'sk KNN classifier accuracy: {sk_knn_clf_accuracy:}')
print(f'sk KNN classifier precision: {sk_knn_clf_precision:}')
print(f'sk KNN classifier recall: {sk_knn_clf_recall:}')
print(f'sk KNN classifier f1: {sk_knn_clf_f1:}')

sk KNN classifier accuracy: 0.5826771653543307
sk KNN classifier precision: 0.6666666666666666
sk KNN classifier recall: 0.6052631578947368
sk KNN classifier f1: 0.6344827586206896


### Обучение модели из sklearn (для регрессии) и оценка качества по выбранным метрикам

Загрузим датасет

In [9]:
df2 = pd.read_csv(dataset_path + "\\Movie_regression.xls")
df2

,Marketing expense,Production expense,Multiplex coverage,Budget,Movie_length,Lead_ Actor_Rating,Lead_Actress_rating,Director_rating,Producer_rating,Critic_rating,Trailer_views,3D_available,Time_taken,Twitter_hastags,Genre,Avg_age_actors,Num_multiplex,Collection
0,20.1264,59.62,0.462,36524.125,138.7,7.825,8.095,7.910,7.995,7.94,527367,YES,109.60,223.840,Thriller,23,494,48000
1,20.5462,69.14,0.531,35668.655,152.4,7.505,7.650,7.440,7.470,7.44,494055,NO,146.64,243.456,Drama,42,462,43200
2,20.5458,69.14,0.531,39912.675,134.6,7.485,7.570,7.495,7.515,7.44,547051,NO,147.88,2022.400,Comedy,38,458,69400
3,20.6474,59.36,0.542,38873.890,119.3,6.895,7.035,6.920,7.020,8.26,516279,YES,185.36,225.344,Drama,45,472,66800
4,21.3810,59.36,0.542,39701.585,127.7,6.920,7.070,6.815,7.070,8.26,531448,NO,176.48,225.792,Drama,55,395,72400
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
501,21.2526,78.86,0.427,36624.115,142.6,8.680,8.775,8.620,8.970,6.80,492480,NO,186.96,243.584,Action,27,561,44800
502,20.9054,78.86,0.427,33996.600,150.2,8.780,8.945,8.770,8.930,7.80,482875,YES,132.24,263.296,Action,20,600,41200
503,21.2152,78.86,0.427,38751.680,164.5,8.830,8.970,8.855,9.010,7.80,532239,NO,109.56,243.824,Comedy,31,576,47800
504,22.1918,78.86,0.427,37740.670,162.8,8.730,8.845,8.800,8.845,6.80,496077,YES,158.80,303.520,Comedy,47,607,44000


Просмотрим информацию о значениях полей для проверки, что все они допустимы

In [10]:
df2.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 506 entries, 0 to 505
Data columns (total 18 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Marketing expense    506 non-null    float64
 1   Production expense   506 non-null    float64
 2   Multiplex coverage   506 non-null    float64
 3   Budget               506 non-null    float64
 4   Movie_length         506 non-null    float64
 5   Lead_ Actor_Rating   506 non-null    float64
 6   Lead_Actress_rating  506 non-null    float64
 7   Director_rating      506 non-null    float64
 8   Producer_rating      506 non-null    float64
 9   Critic_rating        506 non-null    float64
 10  Trailer_views        506 non-null    int64  
 11  3D_available         506 non-null    object 
 12  Time_taken           494 non-null    float64
 13  Twitter_hastags      506 non-null    float64
 14  Genre                506 non-null    object 
 15  Avg_age_actors       506 non-null    int

Удалим некоторые параметры

In [11]:
del df2['3D_available']
del df2['Genre']
del df2['Time_taken']
df2.head()

,Marketing expense,Production expense,Multiplex coverage,Budget,Movie_length,Lead_ Actor_Rating,Lead_Actress_rating,Director_rating,Producer_rating,Critic_rating,Trailer_views,Twitter_hastags,Avg_age_actors,Num_multiplex,Collection
0,20.1264,59.62,0.462,36524.125,138.7,7.825,8.095,7.910,7.995,7.94,527367,223.840,23,494,48000
1,20.5462,69.14,0.531,35668.655,152.4,7.505,7.650,7.440,7.470,7.44,494055,243.456,42,462,43200
2,20.5458,69.14,0.531,39912.675,134.6,7.485,7.570,7.495,7.515,7.44,547051,2022.400,38,458,69400
3,20.6474,59.36,0.542,38873.890,119.3,6.895,7.035,6.920,7.020,8.26,516279,225.344,45,472,66800
4,21.3810,59.36,0.542,39701.585,127.7,6.920,7.070,6.815,7.070,8.26,531448,225.792,55,395,72400


Создадим выборки для обучения и тестирования

In [12]:
X2 = df2.drop('Collection', axis=1)
y2 = df2['Collection']

X2_train,X2_test,y2_train,y2_test = train_test_split(X2.values, y2.values, random_state = 42)

Обучение модели KNN для регрессии

In [13]:
sk_knn_reg = KNeighborsRegressor()
sk_knn_reg.fit(X2_train, y2_train)
sk_knn_reg_pred_res = sk_knn_reg.predict(X2_test)
sk_knn_reg_r2 = r2_score(y2_test, sk_knn_reg_pred_res)
sk_knn_reg_mae = mean_absolute_error(y2_test, sk_knn_reg_pred_res)
sk_knn_reg_mse = mean_squared_error(y2_test, sk_knn_reg_pred_res)

print(f'sk KNN regressor r2: {sk_knn_reg_r2}')
print(f'sk KNN regressor mae: {sk_knn_reg_mae}')
print(f'sk KNN regressor mse: {sk_knn_reg_mse}')

sk KNN regressor r2: 0.6165707960719611
sk KNN regressor mae: 7079.370078740158
sk KNN regressor mse: 108056554.33070867


## 3. Улучшение бейзлайна

### Сформулировать гипотезы (препроцессинг данных, визуализация данных, формирование новых признаков, подбор гиперпараметров на кросс-валидации и т.д.)

1. Формирование новых признаков: для задачи классификации можно создать новые признаки, комбинирующие в себе расходы и рейтинг участвующих в создании фильма людей соответственно.
2. Масштабирование данных во время предобработки
3. Подбор гиперпараметров
    - Количество соседей
    - Весовые функции
    - Метрики расстояния (евклидово, манхэттенское расстояние и т.д.)

#### Задача классификации

1. Формирование новых признаков

In [14]:
data3 = df.copy()
data3["Expense"] = data3["Marketing expense"] + data3["Production expense"]
data3["Rating"] = data3["Lead_ Actor_Rating"] + data3["Lead_Actress_rating"] + data3["Director_rating"] + data3["Producer_rating"]

In [15]:
data4 = data3.copy()
data4 = data4.drop(["Marketing expense","Production expense","Lead_ Actor_Rating","Lead_Actress_rating","Director_rating","Producer_rating"],axis = 1)

In [16]:
X1_new = data4.drop('Start_Tech_Oscar', axis=1)
y1_new = data4['Start_Tech_Oscar']

X1_train_new,X1_test_new,y1_train_new,y1_test_new = train_test_split(X1_new.values, y1_new.values, random_state = 0)

Проверим, как повлияло введение новых признаков

In [17]:
sk_knn_clf = KNeighborsClassifier()
sk_knn_clf.fit(X1_train_new, y1_train_new)
sk_knn_clf_pred_res = sk_knn_clf.predict(X1_test_new)
sk_knn_clf_accuracy = accuracy_score(y1_test_new, sk_knn_clf_pred_res)
sk_knn_clf_precision = precision_score(y1_test_new, sk_knn_clf_pred_res)
sk_knn_clf_recall = recall_score(y1_test_new, sk_knn_clf_pred_res)
sk_knn_clf_f1 = f1_score(y1_test_new, sk_knn_clf_pred_res)

print(f'sk KNN classifier accuracy: {sk_knn_clf_accuracy:}')
print(f'sk KNN classifier precision: {sk_knn_clf_precision:}')
print(f'sk KNN classifier recall: {sk_knn_clf_recall:}')
print(f'sk KNN classifier f1: {sk_knn_clf_f1:}')

sk KNN classifier accuracy: 0.5826771653543307
sk KNN classifier precision: 0.6666666666666666
sk KNN classifier recall: 0.6052631578947368
sk KNN classifier f1: 0.6344827586206896


2. Масштабирование данных

In [18]:
scaler = StandardScaler()
X1_train_scaled = scaler.fit_transform(X1_train)
X1_test_scaled = scaler.transform(X1_test)

Проверим, как повлияло масштабирование данных

In [19]:
sk_knn_clf = KNeighborsClassifier()
sk_knn_clf.fit(X1_train_scaled, y1_train)
sk_knn_clf_pred_res = sk_knn_clf.predict(X1_test_scaled)
sk_knn_clf_accuracy = accuracy_score(y1_test, sk_knn_clf_pred_res)
sk_knn_clf_precision = precision_score(y1_test, sk_knn_clf_pred_res)
sk_knn_clf_recall = recall_score(y1_test, sk_knn_clf_pred_res)
sk_knn_clf_f1 = f1_score(y1_test, sk_knn_clf_pred_res)

print(f'sk KNN classifier accuracy: {sk_knn_clf_accuracy:}')
print(f'sk KNN classifier precision: {sk_knn_clf_precision:}')
print(f'sk KNN classifier recall: {sk_knn_clf_recall:}')
print(f'sk KNN classifier f1: {sk_knn_clf_f1:}')

sk KNN classifier accuracy: 0.5748031496062992
sk KNN classifier precision: 0.6666666666666666
sk KNN classifier recall: 0.5789473684210527
sk KNN classifier f1: 0.619718309859155


3. Подбор гиперпараметров

In [41]:
from sklearn.model_selection import GridSearchCV

param_grid_class = {
    'n_neighbors': [n for n in range(2, 20+1, 1)],
    'weights': ['uniform', 'distance'],
    'metric': ['euclidean', 'manhattan', 'minkowski']
}

knn_class = KNeighborsClassifier()
grid_search_class = GridSearchCV(knn_class, param_grid_class, cv=5, scoring='accuracy')
grid_search_class.fit(X1_train, y1_train)
best_knn_class = grid_search_class.best_estimator_

print("Лучшие параметры для классификации:", grid_search_class.best_params_)

y_pred_class = best_knn_class.predict(X1_test)
accuracy = accuracy_score(y1_test, y_pred_class)
precision = precision_score(y1_test, y_pred_class)
recall = recall_score(y1_test, y_pred_class)
f1 = f1_score(y1_test, y_pred_class)

print(f"Accuracy: {accuracy}")
print(f"Precision: {precision}")
print(f"Recall: {recall}")
print(f"F1 Score: {f1}")

Лучшие параметры для классификации: {'metric': 'euclidean', 'n_neighbors': 3, 'weights': 'distance'}
Accuracy: 0.5748031496062992
Precision: 0.6410256410256411
Recall: 0.6578947368421053
F1 Score: 0.6493506493506495


#### Задача регрессии

2. Масштабирование данных

In [21]:
scaler = StandardScaler()
X2_train_scaled = scaler.fit_transform(X2_train)
X2_test_scaled = scaler.transform(X2_test)

In [22]:
sk_knn_reg = KNeighborsRegressor()
sk_knn_reg.fit(X2_train_scaled, y2_train)
sk_knn_reg_pred_res = sk_knn_reg.predict(X2_test_scaled)
sk_knn_reg_r2 = r2_score(y2_test, sk_knn_reg_pred_res)
sk_knn_reg_mae = mean_absolute_error(y2_test, sk_knn_reg_pred_res)
sk_knn_reg_mse = mean_squared_error(y2_test, sk_knn_reg_pred_res)

print(f'sk KNN regressor r2: {sk_knn_reg_r2}')
print(f'sk KNN regressor mae: {sk_knn_reg_mae}')
print(f'sk KNN regressor mse: {sk_knn_reg_mse}')

sk KNN regressor r2: 0.6770323437261985
sk KNN regressor mae: 5896.377952755905
sk KNN regressor mse: 91017511.81102362


3. Добавим к масштабированию Подбор гиперпараметров

In [40]:
param_grid_reg = {
    'n_neighbors': [n for n in range(1, 20, 1)],
    'weights': ['uniform', 'distance'],
    'metric': ['euclidean', 'manhattan', 'minkowski']
}

knn_reg = KNeighborsRegressor()
grid_search_reg = GridSearchCV(knn_reg, param_grid_reg, cv=5, scoring='r2')
grid_search_reg.fit(X2_train_scaled, y2_train)
best_knn_reg = grid_search_reg.best_estimator_

print("Лучшие параметры для регрессии:", grid_search_reg.best_params_)

y_pred_reg = best_knn_reg.predict(X2_test_scaled)
r2 = r2_score(y2_test, y_pred_reg)
mae = mean_absolute_error(y2_test, y_pred_reg)
mse = mean_squared_error(y2_test, y_pred_reg)

print(f"r2: {r2}")
print("mae:", mae)
print("mse:", mse)

Лучшие параметры для регрессии: {'metric': 'manhattan', 'n_neighbors': 5, 'weights': 'distance'}
r2: 0.72655336788693
mae: 5733.129365870723
mse: 77061685.85171379


### Выводы

Для задачи классификации формирование новых признаков и масштабирование не дали ощутимого улучшения, подбор параметров повысил recall и f1, но понизил accuracy и precision.

Для задачи регрессии масштабирование и подбор параметров позволили однозначно улучшить результат.

## 4. Имплементация алгоритма машинного обучения 

### Самостоятельная имплементация алгоритмов машинного обучения для классификации и регрессии

Реализация для классификации

In [25]:
class KNNClassifier:
    def __init__(self, n_neighbors=5, weights='uniform', metric='minkowski', p=2):
        self.n_neighbors = n_neighbors
        self.weights = weights
        self.metric = metric
        self.p = p

    def fit(self, X, y):
        self.X_train = X
        self.y_train = y

    def predict(self, X):
        predictions = []
        for x_cur in X:
            if self.metric == 'euclidean':
                calculated_distances = np.sqrt(np.sum((self.X_train - x_cur) ** 2, axis=1))
            elif self.metric == 'manhattan':
                calculated_distances = np.sum(np.abs(self.X_train - x_cur), axis=1)
            elif self.metric == 'minkowski':
                calculated_distances = np.linalg.norm(self.X_train - x_cur, ord=self.p, axis=1)
            else:
                raise ValueError("Недопустимый параметр метрики")
            
            closest_indices = np.argsort(calculated_distances)[:self.n_neighbors]
            closest_labels = self.y_train[closest_indices]
            closest_distances = calculated_distances[closest_indices]

            if self.weights == 'uniform':
                weights = np.ones_like(closest_distances)
            elif self.weights == 'distance':
                weights = 1 / (closest_distances + 10**(-5))
            else:
                raise ValueError("Недопустимый параметр метрики")
            
            weighted_votes = {}
            
            for label, weight in zip(closest_labels, weights):
                weighted_votes[label] = weighted_votes.get(label, 0) + weight
            
            prediction = max(weighted_votes, key=weighted_votes.get)
            predictions.append(prediction)
        return np.array(predictions)

Реализация для регрессии

In [26]:
class KNNRegressor:
    def __init__(self, n_neighbors=5, weights='uniform', metric='minkowski', p=2):
        self.n_neighbors = n_neighbors
        self.weights = weights
        self.metric = metric
        self.p = p

    def fit(self, X, y):
        self.X_train = np.array(X, dtype=float)
        self.y_train = np.array(y, dtype=float)


    def predict(self, X):
        X = np.array(X, dtype=float)
        predictions = []
        for x_cur in X:
            x_cur = np.array(x_cur, dtype=float)
            if self.metric == 'euclidean':
                calculated_distances = np.sqrt(np.sum((self.X_train - x_cur) ** 2, axis=1))
            elif self.metric == 'manhattan':
                calculated_distances = np.sum(np.abs(self.X_train - x_cur), axis=1)
            elif self.metric == 'minkowski':
                calculated_distances = np.linalg.norm(self.X_train - x_cur, ord=self.p, axis=1)
            else:
                raise ValueError("Недопустимый параметр метрики")
            
            closest_indices = np.argsort(calculated_distances)[:self.n_neighbors]
            closest_targets = self.y_train[closest_indices]
            closest_distances = calculated_distances[closest_indices]
            
            if self.weights == 'uniform':
                weights = np.ones_like(closest_distances)
            elif self.weights == 'distance':
                weights = 1 / (closest_distances + 10**(-5))
            else:
                raise ValueError("Недопустимый параметр метрики")
            
            weighted_sum = np.sum(weights * closest_targets)
            weighted_mean = weighted_sum / np.sum(weights)
            predictions.append(weighted_mean)
        return np.array(predictions)

### Обучение имплементированных моделей

In [27]:
custom_knn_clf = KNNClassifier(n_neighbors=5, metric='minkowski', weights='distance')
custom_knn_clf.fit(X1_train, y1_train)
custom_knn_clf_pred_res = custom_knn_clf.predict(X1_test)
custom_knn_clf_accuracy = accuracy_score(y1_test, custom_knn_clf_pred_res)
custom_knn_clf_precision = precision_score(y1_test, custom_knn_clf_pred_res)
custom_knn_clf_recall = recall_score(y1_test, custom_knn_clf_pred_res)
custom_knn_clf_f1 = f1_score(y1_test, custom_knn_clf_pred_res)

print(f'custom KNN classifier accuracy: {custom_knn_clf_accuracy:}')
print(f'custom KNN classifier precision: {custom_knn_clf_precision:}')
print(f'custom KNN classifier recall: {custom_knn_clf_recall:}')
print(f'custom KNN classifier f1: {custom_knn_clf_f1:}')

custom KNN classifier accuracy: 0.5984251968503937
custom KNN classifier precision: 0.676056338028169
custom KNN classifier recall: 0.631578947368421
custom KNN classifier f1: 0.6530612244897959


In [28]:
custom_knn_reg = KNNRegressor(n_neighbors=5, metric='manhattan', weights='uniform')
custom_knn_reg.fit(X2_train, y2_train)
custom_knn_reg_pred_res = custom_knn_reg.predict(X2_test)
custom_knn_reg_r2 = r2_score(y2_test, custom_knn_reg_pred_res)
custom_knn_reg_mae = mean_absolute_error(y2_test, custom_knn_reg_pred_res)
custom_knn_reg_mse = mean_squared_error(y2_test, custom_knn_reg_pred_res)

print(f'custom KNN regressor r2: {custom_knn_reg_r2}')
print(f'custom KNN regressor mae: {custom_knn_reg_mae}')
print(f'custom KNN regressor mse: {custom_knn_reg_mse}')

custom KNN regressor r2: 0.6375791402198594
custom KNN regressor mae: 6838.425196850394
custom KNN regressor mse: 102136062.99212599


### Выводы

Полученные результаты схожи с результатами встроенных моделей и даже немного превосходят их, что говорит об отсутствии проблем с реализацией и использованием датасета с этими алгоритмами. 

### Обучение имплементированных моделей в улучшенном бейзлайне

#### Задача классификации

Проверим, как повлияет введение новых признаков

In [29]:
custom_knn_clf = KNNClassifier(n_neighbors=5, metric='minkowski', weights='distance')
custom_knn_clf.fit(X1_train_new, y1_train_new)
custom_knn_clf_pred_res = custom_knn_clf.predict(X1_test_new)
custom_knn_clf_accuracy = accuracy_score(y1_test_new, custom_knn_clf_pred_res)
custom_knn_clf_precision = precision_score(y1_test_new, custom_knn_clf_pred_res)
custom_knn_clf_recall = recall_score(y1_test_new, custom_knn_clf_pred_res)
custom_knn_clf_f1 = f1_score(y1_test_new, custom_knn_clf_pred_res)

print(f'custom KNN classifier accuracy: {custom_knn_clf_accuracy:}')
print(f'custom KNN classifier precision: {custom_knn_clf_precision:}')
print(f'custom KNN classifier recall: {custom_knn_clf_recall:}')
print(f'custom KNN classifier f1: {custom_knn_clf_f1:}')

custom KNN classifier accuracy: 0.5984251968503937
custom KNN classifier precision: 0.676056338028169
custom KNN classifier recall: 0.631578947368421
custom KNN classifier f1: 0.6530612244897959


Проверим, как повлияет масштабирование

In [30]:
custom_knn_clf = KNNClassifier(n_neighbors=5, metric='minkowski', weights='distance')
custom_knn_clf.fit(X1_train_scaled, y1_train)
custom_knn_clf_pred_res = custom_knn_clf.predict(X1_test_scaled)
custom_knn_clf_accuracy = accuracy_score(y1_test, custom_knn_clf_pred_res)
custom_knn_clf_precision = precision_score(y1_test, custom_knn_clf_pred_res)
custom_knn_clf_recall = recall_score(y1_test, custom_knn_clf_pred_res)
custom_knn_clf_f1 = f1_score(y1_test, custom_knn_clf_pred_res)

print(f'custom KNN classifier accuracy: {custom_knn_clf_accuracy:}')
print(f'custom KNN classifier precision: {custom_knn_clf_precision:}')
print(f'custom KNN classifier recall: {custom_knn_clf_recall:}')
print(f'custom KNN classifier f1: {custom_knn_clf_f1:}')

custom KNN classifier accuracy: 0.5669291338582677
custom KNN classifier precision: 0.6567164179104478
custom KNN classifier recall: 0.5789473684210527
custom KNN classifier f1: 0.6153846153846154


Проверим, как повлияет подбор параметров

In [31]:
from itertools import product

best_params_classification = {}
best_metrics_classification = {"accuracy": 0}

n_neighbors_variants = [n for n in range(2, 20+1, 1)]
metric_variants = ['euclidean', 'manhattan', 'minkowski']
weights_variants = ['uniform', 'distance']

for n_neighbors, metric, weights in product(n_neighbors_variants, metric_variants, weights_variants):
    custom_knn_clf = KNNClassifier(n_neighbors=n_neighbors, metric=metric, weights=weights)
    custom_knn_clf.fit(X1_train, y1_train)
    pred_res = custom_knn_clf.predict(X1_test)

    accuracy = accuracy_score(y1_test, pred_res)
    precision = precision_score(y1_test, pred_res)
    recall = recall_score(y1_test, pred_res)
    f1 = f1_score(y1_test, pred_res)

    if accuracy > best_metrics_classification["accuracy"]:
        best_params_classification = {
            "n_neighbors": n_neighbors,
            "metric": metric,
            "weights": weights
        }
        best_metrics_classification = {
            "accuracy": accuracy,
            "precision": precision,
            "recall": recall,
            "f1": f1
        }

print("Лучшие параметры для классификации:", best_params_classification)
print("Метрики для классификации:", best_metrics_classification)

Лучшие параметры для классификации: {'n_neighbors': 14, 'metric': 'manhattan', 'weights': 'distance'}
Метрики для классификации: {'accuracy': 0.6220472440944882, 'precision': 0.6944444444444444, 'recall': 0.6578947368421053, 'f1': 0.6756756756756757}


#### Задача регрессии

Проверим, как повлияет масштабирование

In [32]:
custom_knn_reg = KNNRegressor(n_neighbors=5, metric='manhattan', weights='uniform')
custom_knn_reg.fit(X2_train_scaled, y2_train)
custom_knn_reg_pred_res = custom_knn_reg.predict(X2_test_scaled)
custom_knn_reg_r2 = r2_score(y2_test, custom_knn_reg_pred_res)
custom_knn_reg_mae = mean_absolute_error(y2_test, custom_knn_reg_pred_res)
custom_knn_reg_mse = mean_squared_error(y2_test, custom_knn_reg_pred_res)

print(f'custom KNN regressor r2: {custom_knn_reg_r2}')
print(f'custom KNN regressor mae: {custom_knn_reg_mae}')
print(f'custom KNN regressor mse: {custom_knn_reg_mse}')

custom KNN regressor r2: 0.724920866424201
custom KNN regressor mae: 5817.007874015748
custom KNN regressor mse: 77521751.18110237


Добавим к масштабированию подбор параметров

In [33]:
best_params_regression = {}
best_metrics_regression = {"r2": 0}

n_neighbors_variants = [n for n in range(2, 20+1, 1)]
metric_variants = ['euclidean', 'manhattan', 'minkowski']
weights_variants = ['uniform', 'distance']

for n_neighbors, metric, weights in product(n_neighbors_variants, metric_variants, weights_variants):
    custom_knn_reg = KNNRegressor(n_neighbors=n_neighbors, metric=metric, weights=weights)
    custom_knn_reg.fit(X2_train_scaled, y2_train)
    pred_res = custom_knn_reg.predict(X2_test_scaled)

    r2 = r2_score(y2_test, pred_res)
    mae = mean_absolute_error(y2_test, pred_res)
    mse = mean_squared_error(y2_test, pred_res)

    if r2 > best_metrics_regression["r2"]:
        best_params_regression = {
            "n_neighbors": n_neighbors,
            "metric": metric,
            "weights": weights
        }
        best_metrics_regression = {
            "r2": r2,
            "mae": mae,
            "mse": mse
        }

print("Лучшие параметры для регрессии:", best_params_regression)
print("Метрики для регрессии:", best_metrics_regression)

Лучшие параметры для регрессии: {'n_neighbors': 5, 'metric': 'manhattan', 'weights': 'distance'}
Метрики для регрессии: {'r2': 0.7265533617721194, 'mae': 5733.129644356171, 'mse': 77061687.57496642}


### Улучшенный бейзлайн

In [34]:
custom_knn_clf = KNNClassifier(n_neighbors=14, metric='manhattan', weights='distance')
custom_knn_clf.fit(X1_train, y1_train)
pred_res = custom_knn_clf.predict(X1_test)

accuracy = accuracy_score(y1_test, pred_res)
precision = precision_score(y1_test, pred_res)
recall = recall_score(y1_test, pred_res)
f1 = f1_score(y1_test, pred_res)

print(f'custom KNN classifier accuracy: {accuracy:}')
print(f'custom KNN classifier precision: {precision:}')
print(f'custom KNN classifier recall: {recall:}')
print(f'custom KNN classifier f1: {f1:}')

custom KNN classifier accuracy: 0.6220472440944882
custom KNN classifier precision: 0.6944444444444444
custom KNN classifier recall: 0.6578947368421053
custom KNN classifier f1: 0.6756756756756757


In [35]:
custom_knn_reg = KNNRegressor(n_neighbors=5, metric='manhattan', weights='distance')
custom_knn_reg.fit(X2_train_scaled, y2_train)
custom_knn_reg_pred_res = custom_knn_reg.predict(X2_test_scaled)
custom_knn_reg_r2 = r2_score(y2_test, custom_knn_reg_pred_res)
custom_knn_reg_mae = mean_absolute_error(y2_test, custom_knn_reg_pred_res)
custom_knn_reg_mse = mean_squared_error(y2_test, custom_knn_reg_pred_res)

print(f'custom KNN regressor r2: {custom_knn_reg_r2}')
print(f'custom KNN regressor mae: {custom_knn_reg_mae}')
print(f'custom KNN regressor mse: {custom_knn_reg_mse}')

custom KNN regressor r2: 0.7265533617721194
custom KNN regressor mae: 5733.129644356171
custom KNN regressor mse: 77061687.57496642


### Выводы

Как и для встроенных моделей, удалось улучшить результаты работы моделей.
В целом, результаты встроеных и собственных моделей в улучшенных бейзлайнах схожи, для классификации некоторые метрики лучше у встроенной модели, некоторые - у сосбтвенной.